## Check your data before end-users use it, with the WAP pattern

* It is extremely hard (and at times impossible) to recover from the impact of bad data. Businesses making a decision based on incorrect data, sending your data out to a 3-rd party, etc.
* Data quality is becoming increasingly important as the need to make data available faster grows.
* To ensure data correctness, our pipelines need to quality-check the data before it is made available to users.
* This is where the `Write-Audit-Publish (WAP)` pattern comes into play.
* In WAP, we first create a temporary representation of the data (e.g., a dataframe or temporary table), check that it meets our data quality requirements, and load it into the destination table, which users can access only if it passes.

```mermaid
flowchart TD
    A[Genrate data] -->B[Check data quality]
    B --> C[Log the results of the check]
    C --> D{Did the data pass the check}
    D -->|Yes| E[Write the data to its storage location]
    D -->|No| F[Raise an alert and warn DEs]
    E --> G[Ready for downstream consumers]
    F --> H[Fix the issue]
    H --> A
```

* We should also track the results of the DQ checks over time, which will provide us with insights into which checks fail most often and help us prioritize fixing them.

#### Example

* Consider the code below, that creates `dim_customer` table.
* Assume that we need to check that the email column does not contain null values.
* Let’s see how to implement this.

In [ ]:
import argparse

from pyspark.sql import DataFrame, SparkSession

TABLE_NAME = "local.silver.dim_customer"


def extract(spark: SparkSession) -> dict[str, DataFrame]:
    customer_df = spark.table("local.bronze.customer")
    customer_address_df = spark.table("local.bronze.customer_address")
    return {"customer_df": customer_df, "customer_address_df": customer_address_df}


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    input_dfs["customer_df"].createOrReplaceTempView("customer")
    input_dfs["customer_address_df"].createOrReplaceTempView("customer_address")

    return spark.sql("""
        SELECT
            c.customer_id,
            c.email,
            c.full_name,
            c.phone,
            c.status,
            c.created_at,
            c.updated_at,
            COLLECT_LIST(
                STRUCT(
                    ca.is_default,
                    CONCAT(ca.line1, ', ', ca.city, ', ', ca.state, ', ', ca.country) AS address
                )
            ) AS addresses
        FROM customer c
        LEFT JOIN customer_address ca USING (customer_id)
        GROUP BY 1, 2, 3, 4, 5, 6, 7
    """)


def load(output_df: DataFrame) -> None:
    output_df.writeTo(TABLE_NAME).createOrReplace()


def validate(transformed_df: DataFrame) -> bool:
    return transformed_df["email"].notna().all()


def run(spark: SparkSession) -> None:
    transformed_df = transform(extract(spark))
    if not validate(transformed_df):
        raise ValueError("Data Quality Check Failed")
    load(transformed_df)

* We create a `validate` function that checks the data quality and returns a boolean. Our `run` function uses this to raise an exception if it notices an error.

#### Exercise [10 min] 

In addition to the `email is not null` check above. Include the following checks.

1. updated_at >= created_at
2. status must be one of ['inactive', 'active', 'suspended']


* We check for DQ, but we do not compute or log details. We will see how to do this in a later chapter

## Choose the type of data quality check based on the data

* There are an infinite number of data quality checks.
* However, creating too many data quality checks can lead to noisy alerts and overwhelm data engineers.
* Shown below are the main types of data quality checks
  * **Table and column constraints:** This includes checking if the data schema matches an expected schema. Column-level checks like uniqueness, not nulls, enums, etc
  * **Business rules:** These checks ensure the data aligns with what your business expects. E.g., range of acceptable values, debit should always be positive, and the purchase date should precede the ship date.
  * **Metric checks:** For the key metrics of your data (e.g., revenue, conversion rate, etc.), checks such as anomaly detection and variance checks (run-over-run change should be less than a specific threshold) are critical.
  * **Referential Integrity/Relationship Checks:** When modeling data, we want to ensure that our output is not only complete but also usable across other tables. Referential integrity here refers to an ID that is complete in another table.
    - For example, if a fact table has a dimension ID that has not yet been loaded into its dimension table, we should be aware of this, or we will get a faulty join.
  * **Reconciliation Checks:** For critical metrics or events (e.g., orders), we may want to ensure the output aligns with the input.
    - This can be checking that the number of rows of the output is close to the number of rows of the input (to catch faulty joins/filters in our pipeline), or checking that the sum/avg of some metric is the same between the output and the inputs

![Reconciliation Check](images/reconciliation_check.png)

* Metric and reconciliation checks are often also done after aggregating by dimensions. For example, compare state-level revenue to previous runs. This helps account for dimensional skew (e.g., CA revenue is much larger than in other states in the US). This will also help us isolate issues more quickly.

| dimension | metric         | current_run | previous_run | delta  | delta_pct | status |
|-----------|----------------|-------------|--------------|--------|-----------|--------|
| CA        | revenue        | 1,200,000   | 1,150,000    | 50,000 | +4.3%     | ✅ PASS |
| TX        | revenue        | 450,000     | 448,000      | 2,000  | +0.4%     | ✅ PASS |
| NY        | revenue        | 380,000     | 310,000      | 70,000 | +22.6%    | ❌ FAIL |
| FL        | revenue        | 210,000     | 208,000      | 2,000  | +1.0%     | ✅ PASS |
| WA        | revenue        | 5,000       | 195,000      | -190,000 | -97.4%  | ❌ FAIL |

#### Exercise [5 min]

Assume you have a summary table that has the following schema.

1. Order_month
2. State_name
3. number_of_orders
4. Total_revenue

What are the top 3 types of DQ checks that you would implement?

## Implementing DQ checks

* Implementing DQ checks is a lot of work. You need to ensure that the validation logic is accurate and handles the storage of results.
* There are multiple tools to help you run DQ checks and log their results. A few popular ones are dbt-expectations, great expectations, soda-core, etc.
* Most DQ frameworks have 3 parts.
  1. `DQ check definition`: Defining the DQ checks to be run as a json or yaml of Python code
  2. Code to run `DQ check` & log statistics about the result of the DQ checks
  3. A way to `store the output`
* Let’s use soda-core to set up DQ checks for our data pipelines.

##### Example

* Let’s set up DQ checks for dim_customer to check that.
  * customer ID has no nulls
  * The email column follows the standard email format
  * row count reconciliation check with local.bronze.customer table
* First, let’s define the DQ checks as a yaml file: [dim_customer_dq.yaml](./dim_customer_dq.yaml)
* Let’s use the `soda-core` Python library to verify that dim_customer passes these checks.

In [ ]:
import pprint

from pyspark.sql import Row, SparkSession
from soda_core.contracts import verify_contract_locally
from soda_sparkdf import SparkDataFrameDataSource

# Create Spark session
spark = SparkSession.builder.appName("04_data_quality").master("local[*]").getOrCreate()

# Create a temp view corresponding to the yaml
spark.table("local.silver.dim_customer").createOrReplaceTempView("tmp_customer")

spark_data_source = SparkDataFrameDataSource.from_existing_session(
    session=spark, name="my_sparkdf"
)

result = verify_contract_locally(
    data_sources=[spark_data_source], contract_file_path="./dim_customer_dq.yaml"
)

if result.is_ok:
    print("✅ Contract verification passed.")
else:
    print("❌ Contract verification failed:")
    print(result.get_errors_str())

pprint.pp(
    [c.log_table_row() for c in result.contract_verification_results[0].check_results]
)

#### Exercise [10 min]

Create the following DQ checks for local.silver.fct_order_lines
  - reconciliation count with `local.bronze.order_line`,
  - check no nulls in the order_id column

* create a file at [fct_order_lines.yaml](./fct_order_lines.yaml) to define the data quality check. Use [these docs](https://docs.soda.io/reference/contract-language-reference) to identify the correct format.

In [ ]:
%%bash
%%capture
echo 'RUN Bronze local.bronze.order_lines'
echo '================='
uv run python ./bronze/order_lines.py

echo 'RUN Silver local.silver.fct_order_lines'
echo '================='
uv run python ./silver/fct_order_lines.py --start-time '2025-01-01 00:00:00' --end-time '2026-01-01 00:00:00'

In [ ]:
# this should work

import pprint

from pyspark.sql import SparkSession
from soda_core.contracts import verify_contract_locally
from soda_sparkdf import SparkDataFrameDataSource

spark.table("local.silver.fct_order_lines").createOrReplaceTempView(
    "tmp_fct_order_lines"
)

spark_data_source = SparkDataFrameDataSource.from_existing_session(
    session=spark, name="my_sparkdf"
)

result = verify_contract_locally(
    data_sources=[spark_data_source], contract_file_path="./fct_order_lines.yaml"
)

if result.is_ok:
    print("✅ Contract verification passed.")
else:
    print("❌ Contract verification failed:")
    print(result.get_errors_str())

pprint.pp(
    [c.log_table_row() for c in result.contract_verification_results[0].check_results]
)

* DQ tools are powerful, but they are not always optimized for performance.
* A big issue with DQ tools are that they run a separate query for each DQ check, sometimes resulting in multiple table scans, be mindful of this for the DQ tool that you use
* In SODA we can alleviate this by using a SQL based check combining multiple checks, the downside is that the specificity of results will not be good and you will need to handle this (either by printing some failed rows, etc)

**In-Pipeline v Out-of-Pipeline checks**

* Most DQ checks are hard blockers, meaning if they fail the data is unusable
* There is another pattern of DQ checks that ;are meant to alert the data engineer of a potential problem but not important enough to block a data from being released, these are called out-of-pipeline or async dq checks
* Some examples of this are data distribution/metric skew, data schema changes over time

#### Example

* Assume you have a data pipeline that is supposed to arrive once every 6h, what async dq check will you run (& how ) to check that this is the case?
* check time delay between event creation time and current time and get the max of that value

## Recap

In this section, we covered data quality. Data quality is a bottleneck for most data engineering teams. We covered the following.

1. Write-Audit-Publish WAP implementation
2. Reducing alert noise by choosing the most appropriate DQ checks
3. Ways to implement DQ checks

These techniques will enable you to build data products that earn the end-user's trust.